# 01 — Data Cleaning
Load the UCI Bank Marketing dataset, handle missing values, encode categoricals, treat outliers, and save a clean CSV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

RAW_PATH  = '../data/raw/bank-additional-full.csv'
SAVE_PATH = '../data/processed/cleaned_data.csv'
os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv(RAW_PATH, sep=';')
print(f'Shape: {df.shape}')
df.head()

## 1. Basic Info & Missing Values

In [ ]:
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nValue counts for "unknown" entries:')
for col in df.select_dtypes('object').columns:
    unk = (df[col] == 'unknown').sum()
    if unk > 0:
        print(f'  {col}: {unk} unknowns ({unk/len(df)*100:.1f}%)')

## 2. Handle 'unknown' Values

In [ ]:
# Replace 'unknown' with NaN, then impute with mode for low-missing cols
df.replace('unknown', np.nan, inplace=True)

low_miss_cols = ['job', 'marital', 'education']
for col in low_miss_cols:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
    print(f'{col} filled with mode: {mode_val}')

# Drop rows where default / housing / loan still missing (small %)
before = len(df)
df.dropna(subset=['default', 'housing', 'loan'], inplace=True)
print(f'\nDropped {before - len(df)} rows with missing default/housing/loan')
print(f'Remaining: {len(df)}')

## 3. Target Encoding

In [ ]:
df['y'] = (df['y'] == 'yes').astype(int)
print('Target distribution:')
print(df['y'].value_counts())
print(f'Overall conversion rate: {df["y"].mean()*100:.2f}%')

## 4. Fix Data Types & Encode Categoricals

In [ ]:
# Binary columns
binary_map = {'yes': 1, 'no': 0}
for col in ['default', 'housing', 'loan']:
    df[col] = df[col].map(binary_map)

# Month → numeric order
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
df['month_num'] = df['month'].map(month_map)

# Day of week
day_map = {'mon':1,'tue':2,'wed':3,'thu':4,'fri':5}
df['day_num'] = df['day_of_week'].map(day_map)

# One-hot encode job, marital, education, contact, poutcome
df = pd.get_dummies(df, columns=['job','marital','education','contact','poutcome'], drop_first=False)
print(f'Shape after encoding: {df.shape}')

## 5. Outlier Treatment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['duration', 'campaign', 'previous']):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
plt.suptitle('Outlier Check (before capping)')
plt.tight_layout()
plt.show()

# Cap at 99th percentile
for col in ['duration', 'campaign', 'previous']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)
    print(f'{col} capped at {cap:.0f}')

## 6. Remove Duplicates & Save

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

df.to_csv(SAVE_PATH, index=False)
print(f'\nCleaned data saved → {SAVE_PATH}')
print(f'Final shape: {df.shape}')